# Phase 1.4 — Data Quality

This notebook checks the Retailrocket dataset for data-quality issues and establishes practical cleaning rules before downstream dataset construction.


In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "data" / "raw").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
EVENTS_PATH = RAW_DIR / "events.csv"
PROPERTIES_1_PATH = RAW_DIR / "item_properties_part1.csv"
PROPERTIES_2_PATH = RAW_DIR / "item_properties_part2.csv"
CATEGORY_PATH = RAW_DIR / "category_tree.csv"

print("Project root:", PROJECT_ROOT)
print("Raw directory:", RAW_DIR)


Project root: f:\annuspeaks.com\recommendation-system
Raw directory: f:\annuspeaks.com\recommendation-system\data\raw


## 1. Missing Values

Check missing values in the behavioral interaction and product metadata files.


In [2]:
# Detect missing values

files = {
    "events": (EVENTS_PATH, ["timestamp", "visitorid", "event", "itemid", "transactionid"]),
    "item_properties_part1": (PROPERTIES_1_PATH, ["timestamp", "itemid", "property", "value"]),
    "item_properties_part2": (PROPERTIES_2_PATH, ["timestamp", "itemid", "property", "value"]),
    "category_tree": (CATEGORY_PATH, None),
}

for name, (path, usecols) in files.items():
    df = pd.read_csv(path, usecols=usecols)
    print(f"{name}:")
    display(df.isna().sum())
    print()


events:


timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644
dtype: int64


item_properties_part1:


timestamp    0
itemid       0
property     0
value        0
dtype: int64


item_properties_part2:


timestamp    0
itemid       0
property     0
value        0
dtype: int64


category_tree:


categoryid     0
parentid      25
dtype: int64

## 2. Duplicate Interactions

Detect exact duplicate behavioral interaction records.


In [3]:
# Detect duplicate interactions

duplicate_count = 0
total_events = 0

for chunk in pd.read_csv(
    EVENTS_PATH,
    usecols=["timestamp", "visitorid", "event", "itemid", "transactionid"],
    chunksize=250_000,
):
    total_events += len(chunk)
    duplicate_count += int(chunk.duplicated().sum())

print("Total event rows:", f"{total_events:,}")
print("Exact duplicate rows detected within processed chunks:", f"{duplicate_count:,}")
print("Note: this checks exact duplicates within each chunk.")


Total event rows: 2,756,101
Exact duplicate rows detected within processed chunks: 441
Note: this checks exact duplicates within each chunk.


## 3. Invalid Product / User IDs

Check whether core user and product identifiers are missing or invalid.


In [4]:
# Detect invalid product/user IDs

invalid_user_ids = 0
invalid_item_ids = 0
event_rows = 0

for chunk in pd.read_csv(
    EVENTS_PATH,
    usecols=["visitorid", "itemid"],
    chunksize=250_000,
):
    event_rows += len(chunk)
    invalid_user_ids += int(
        chunk["visitorid"].isna().sum() + (chunk["visitorid"] < 0).sum()
    )
    invalid_item_ids += int(
        chunk["itemid"].isna().sum() + (chunk["itemid"] < 0).sum()
    )

print("Rows checked:", f"{event_rows:,}")
print("Invalid user IDs:", f"{invalid_user_ids:,}")
print("Invalid product IDs:", f"{invalid_item_ids:,}")


Rows checked: 2,756,101
Invalid user IDs: 0
Invalid product IDs: 0


## 4. Inconsistent Metadata

Inspect the generic product-property structure for missing fields and conflicting values.


In [5]:
# Detect inconsistent metadata

metadata_rows = 0
missing_metadata_rows = 0
conflict_groups = 0

for path in [PROPERTIES_1_PATH, PROPERTIES_2_PATH]:
    for chunk in pd.read_csv(
        path,
        usecols=["itemid", "property", "value"],
        chunksize=250_000,
    ):
        metadata_rows += len(chunk)
        missing_metadata_rows += int(
            chunk[["itemid", "property", "value"]].isna().any(axis=1).sum()
        )

        conflicts = (
            chunk.dropna(subset=["itemid", "property", "value"])
            .groupby(["itemid", "property"])["value"]
            .nunique()
        )
        conflict_groups += int((conflicts > 1).sum())

print("Metadata rows checked:", f"{metadata_rows:,}")
print("Rows with missing metadata fields:", f"{missing_metadata_rows:,}")
print("Item/property groups with multiple values within processed chunks:", f"{conflict_groups:,}")
print("Property semantics remain generic until product-dataset mapping.")


Metadata rows checked: 20,275,902
Rows with missing metadata fields: 0
Item/property groups with multiple values within processed chunks: 507,332
Property semantics remain generic until product-dataset mapping.


## 5. Anomalous Ratings / Interactions

Retailrocket provides behavioral events rather than explicit user ratings. Check event types and timestamp validity.


In [6]:
# Analyze anomalous ratings/interactions

event_counts = {}
invalid_timestamps = 0
total_rows = 0

for chunk in pd.read_csv(
    EVENTS_PATH,
    usecols=["timestamp", "event"],
    chunksize=250_000,
):
    total_rows += len(chunk)

    for event, count in chunk["event"].value_counts(dropna=False).items():
        event_counts[event] = event_counts.get(event, 0) + int(count)

    timestamps = pd.to_datetime(chunk["timestamp"], unit="ms", errors="coerce")
    invalid_timestamps += int(timestamps.isna().sum())

print("Event types:")
print(pd.Series(event_counts).sort_index())
print("\nInvalid timestamps:", f"{invalid_timestamps:,}")
print("Explicit ratings detected: No — Retailrocket events are behavioral interactions.")


Event types:
addtocart        69332
transaction      22457
view           2664312
dtype: int64

Invalid timestamps: 0
Explicit ratings detected: No — Retailrocket events are behavioral interactions.


## 6. Data-Cleaning Rules

The following rules will guide downstream processing:

- Preserve original files under `data/raw/`.
- Never modify raw source files.
- Keep valid user/item identifiers.
- Missing `transactionid` is acceptable for non-purchase events.
- Treat `view`, `addtocart`, and `transaction` as implicit behavioral feedback.
- Remove exact duplicate interaction records when constructing canonical processed data.
- Keep product metadata in generic `property` / `value` form until semantic mappings are established.
- Handle invalid/missing required identifiers during processed-dataset construction.


In [7]:
# Final Phase 1.4 quality summary

checks = [
    "Missing values",
    "Duplicate interactions",
    "Invalid product/user IDs",
    "Inconsistent metadata",
    "Anomalous ratings/interactions",
    "Data-cleaning rules",
]

print("DATA QUALITY CHECK COMPLETE")
for i, check in enumerate(checks, 1):
    print(f"{i}. {check}")


DATA QUALITY CHECK COMPLETE
1. Missing values
2. Duplicate interactions
3. Invalid product/user IDs
4. Inconsistent metadata
5. Anomalous ratings/interactions
6. Data-cleaning rules
